In [35]:
import pickle
import os
from pathlib import Path

import sympy as sp
import sympy.physics.mechanics as mech
from IPython.display import display, Math
sp.init_printing()
DIR = Path(os.getcwd())

# Double pendulum derivation

<center>
<img src='./media/cartpole-cart-double.svg' style='width: 30%; height: auto;' alt='Single pendulum'>
</center>

In [36]:
m, M, g, l_1, l_2, m_1, m_2, J_1, J_2 = sp.symbols('m M g l_1 l_2 m_1 m_2 J_1 J_2')

t = mech.dynamicsymbols._t

q1, q2, q3 = mech.dynamicsymbols('x theta_1 theta_2')
q1d, q2d, q3d = mech.dynamicsymbols('x theta_1 theta_2', 1)

u, w = mech.dynamicsymbols('u w')   # control action and disturbance

# Inertial frame
N = mech.ReferenceFrame('N')

# r1_frame.y points along rod of theta1, rotate inertial frame q2 about N.z
r1_frame = N.orientnew("R1", "Axis", (q2, N.z))
# r2_frame.y points along rod of theta1+theta2, rotate inertial frame q2+q3 about N.z
r2_frame = r1_frame.orientnew("R2", "Axis", (q3, r1_frame.z))


# Origin
O = mech.Point('O')
O.set_vel(N, 0)

# Cart point (COM for M)
P_M = O.locatenew('P_M', q1*N.x)
P_M.set_vel(N, q1d*N.x)

# Rod 1 center of mass, halfway along rod
P_R1 = P_M.locatenew("P_R1", (l_1/2)*r1_frame.y)
P_R1.set_vel(N, P_R1.pos_from(O).dt(N))

# Rod 2 center of mass, halfway along rod
P_R2 = P_M.locatenew("P_R2", l_1*r1_frame.y + l_2/2*r2_frame.y)
P_R2.set_vel(N, P_R2.pos_from(O).dt(N))

# Pendulum end point (COM for m)
P_m = P_M.locatenew(
    'P_m',
    l_1*r1_frame.y + l_2*r2_frame.y
)
P_m.set_vel(N, P_m.pos_from(O).dt(N))


rod1_inertia = mech.inertia(r1_frame, 0, 0, J_1)
rod1 = mech.RigidBody("rod1", P_R1, r1_frame, m_1, (rod1_inertia, P_R1))

rod2_inertia = mech.inertia(r2_frame, 0, 0, J_2)
rod2 = mech.RigidBody("rod2", P_R2, r2_frame, m_2, (rod2_inertia, P_R2))

# Particles
cart = mech.Particle('cart', P_M, M)
pole_end = mech.Particle('bob', P_m, m)
bodies = [cart, rod1, rod2, pole_end]

# Potential energy
cart.potential_energy = 0
rod1.potential_energy = m_1*g*P_R1.pos_from(O).express(N).dot(N.y)
rod2.potential_energy = m_2*g*P_R2.pos_from(O).express(N).dot(N.y)
pole_end.potential_energy = m*g*P_m.pos_from(O).express(N).dot(N.y)


# Lagrangian
L = mech.Lagrangian(N, *bodies)

# Non-conservative force on cart
forcelist = [
    (P_M, (u+w)*N.x)  # control u
]

LM = mech.LagrangesMethod(
    L,
    [q1, q2, q3],
    forcelist=forcelist,
    frame=N,
    bodies=bodies
)

eom = LM.form_lagranges_equations()
z = LM.q.col_join(LM._qdots)    # sympy stores state as q, q'

In [37]:
def display_sym_expr(expr: sp.Basic, size: str='Large'):
    # Symbols for printed / state-space form. Remove the explicit time dependence x(t) -> x, and
    # replace d/dt notation with dots
    x_s = sp.Symbol('x')
    theta1_s = sp.Symbol(r'\theta_1')
    theta2_s = sp.Symbol(r'\theta_2')

    xd_s = sp.Symbol(r'\dot{x}')
    theta1d_s = sp.Symbol(r'\dot\theta_1')
    theta2d_s = sp.Symbol(r'\dot\theta_2')

    xdd_s = sp.Symbol(r'\ddot{x}')
    theta1dd_s = sp.Symbol(r'\ddot\theta_1')
    theta2dd_s = sp.Symbol(r'\ddot\theta_2')

    # Important: substitute higher derivatives first
    pretty_subs = [
        (sp.diff(q1, t, 2), xdd_s),
        (sp.diff(q2, t, 2), theta1dd_s),
        (sp.diff(q3, t, 2), theta2dd_s),
        (sp.diff(q1, t), xd_s),
        (sp.diff(q2, t), theta1d_s),
        (sp.diff(q3, t), theta2d_s),
        (q1, x_s),
        (q2, theta1_s),
        (q3, theta2_s),
    ]
    expr_pretty = expr.subs(pretty_subs)
    expr_pretty = expr_pretty
    display(Math(rf'\{size} ' + sp.latex(expr_pretty)))

In [38]:
print('Implicit EOM:')
display_sym_expr(eom)

print('Mass matrix:')
display_sym_expr(LM.mass_matrix)

print('Forcing:')
display_sym_expr(LM.forcing)

print('Explicit EOMs')
rhs_exp = LM.rhs() # dont simplify, takes very long
# display_sym_expr(sp.Equality(sp.diff(z, t), rhs_exp))

Implicit EOM:


<IPython.core.display.Math object>

Mass matrix:


<IPython.core.display.Math object>

Forcing:


<IPython.core.display.Math object>

Explicit EOMs


### Linearized dynamics

##### We are not interested in the full linearization expression about some linearization point, only the derivatives with respect to our state $z$ and control action $u$.

In [39]:
# Reorder variables as z = x, x', theta, theta' to agree with function calls in dynamics.py
P = sp.Matrix([
    [1, 0, 0, 0, 0, 0],  # x
    [0, 0, 0, 1, 0, 0],  # x_dot
    [0, 1, 0, 0, 0, 0],  # theta_1
    [0, 0, 0, 0, 1, 0],  # theta_1_dot
    [0, 0, 1, 0, 0, 0],  # theta_2
    [0, 0, 0, 0, 0, 1],  # theta_2_dot
])

z = P * z
f = P * rhs_exp

df_dz = f.jacobian(z)
df_du = f.jacobian([u])

display(Math(r'$\Large z$'))
display_sym_expr(z)
# display(Math(r'$\Large f, \quad \dot z = f(z, \dot z)$'))
# display_sym_expr(f)
# display(Math(r'$\Large \frac {df} {dz}$'))
# display_sym_expr(df_dz)
# display(Math(r'$\Large \frac {df} {du}$'))
# display_sym_expr(df_du)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

##### We are also interested in the energy in the system

In [40]:
T = mech.kinetic_energy(N, *LM.bodies)
V = mech.potential_energy(*LM.bodies)


print('Kinetic energy')
display_sym_expr(T)
print('Potential energy')
display_sym_expr(V)

Kinetic energy


<IPython.core.display.Math object>

Potential energy


<IPython.core.display.Math object>

### Export dynamics

##### Convert the dynamic symbols (time dependent functions) and replace them by a symbol. We want to export the formulas with non-time dependent state symbols, as we want to evaluate the expression at some value of z, u, etc. and not take time into consideration

In [41]:
# this should cover all of the dynamic symbols
x_s, x_t_s, x_tt_s = sp.symbols('x x_t x_tt')
th1_s, th1_t_s, th1_tt_s = sp.symbols(r'\theta_1 \theta_{1t} \theta_{1tt}')
th2_s, th2_t_s, th2_tt_s = sp.symbols(r'\theta_2 \theta_{2t} \theta_{2tt}')
u_s, w_s = sp.symbols('u w')
dyn_to_static = {
    q1.diff(t, 2): x_tt_s,
    q2.diff(t, 2): th1_tt_s,
    q3.diff(t, 2): th2_tt_s,
    q1.diff(t): x_t_s,
    q2.diff(t): th1_t_s,
    q3.diff(t): th2_t_s,
    q1: x_s,
    q2: th1_s,
    q3: th2_s,
    u: u_s,
    w: w_s,
}

def get_symbols(expr: sp.Basic):
    return expr.free_symbols, mech.find_dynamicsymbols(expr)


all_symbols = set()

exported_expr = {'f': f, 'df_dz': df_dz, 'df_du': df_du, 'Ek': T, 'Ep': V, 'state_symbols': z,
    'state_dot_symbols': sp.diff(z, t), 'u_symbol': u, 'w_symbol': w}

for expr_name in exported_expr.keys():
    exported_expr[expr_name] = exported_expr[expr_name].subs(dyn_to_static)
    all_symbols.update(*get_symbols(exported_expr[expr_name]))


exported_expr['func_symbols'] = all_symbols

assert t not in all_symbols, 'Failed to remove all time dependency from sympy expressions'    

with open(DIR / 'cart_pole' / 'dynamics_double.pkl', 'wb') as outfile:
    pickle.dump(exported_expr, outfile)


for (expr_name, expr) in exported_expr.items():
    print(f'Free and dynamic symbols in {expr_name}')
    try:
        display(get_symbols(expr))
    except:
        display(expr)



Free and dynamic symbols in f


Free and dynamic symbols in df_dz


Free and dynamic symbols in df_du


Free and dynamic symbols in Ek


Free and dynamic symbols in Ep


Free and dynamic symbols in state_symbols


Free and dynamic symbols in state_dot_symbols


Free and dynamic symbols in u_symbol


Free and dynamic symbols in w_symbol


Free and dynamic symbols in func_symbols
